# DiffKilter: Model Training

This notebook trains a transformer using Discrete Absorbing Diffusion (based on D3PM) to generate climbing routes on the Kilterboard.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np
from tqdm.auto import tqdm

from dataset import KilterDataset
from model import KilterTransformer
from diffusion import get_noise_schedule, apply_absorbing_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
BATCH_SIZE = 128
EPOCHS = 128
LR = 1e-4
NUM_TIMESTEPS = 100
MASK_TOKEN_ID = 5

#  Load dataset
dataset = KilterDataset('./data/kilter_board_climbs.npz')
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

alphas_cumprod = get_noise_schedule(num_timesteps=NUM_TIMESTEPS, device=device)

print(f"Loaded {len(dataset)} routes.")

In [ ]:
alpha_weights = torch.tensor([0.6, 0.5, 0.5, 0.5, 0.5, 0.0])

class FocalLoss(nn.Module):
  def __init__(self, gamma=2.0, alpha=alpha_weights, reduction='mean'):
    super(FocalLoss, self).__init__()
    self.gamma = gamma
    self.alpha = alpha
    self.reduction = reduction

  def forward(self, inputs, targets):

    ce_loss = F.cross_entropy(inputs, targets, reduction='none')
    pt = torch.exp(-ce_loss)
    focal_term = (1 - pt) ** self.gamma

    if self.alpha is not None:
      if isinstance(self.alpha, (list, tuple, torch.Tensor)):
        alpha_t = self.alpha[targets]
        loss = alpha_t * focal_term * ce_loss
      else:
        loss = self.alpha * focal_term * ce_loss
    else:
      loss = focal_term * ce_loss

    if self.reduction == 'mean':
      return loss.mean()
    elif self.reduction == 'sum':
      return loss.sum()
    else:
      return loss

In [ ]:
model = KilterTransformer(
    num_classes=6,
    hidden_dim=256,
    num_layers=6,
    nhead=8
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4) #  Maybe remove weight decay if climbs are crap
criterion = FocalLoss()

In [ ]:
save_dir = './weights'
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  Initialise dataset, dataloader, model, optimizer, criterion
resume_checkpoint_path = f"{save_dir}/kilter_checkpoint_epoch_100.pt"
start_epoch = 0

if os.path.exists(resume_checkpoint_path):
    print(f"Resuming from checkpoint: {resume_checkpoint_path}")
    checkpoint = torch.load(resume_checkpoint_path, map_location=device)
    start_epoch = checkpoint['epoch'] + 1
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Resuming at Epoch {start_epoch}")
else:
    print("Starting training from scratch.")

num_epochs = 110
num_timesteps = 100

In [ ]:
#  The training loop
for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_routes, batch_coords in pbar:
        batch_routes = batch_routes.to(device)
        batch_coords = batch_coords.to(device)

        optimizer.zero_grad()

        t = torch.randint(1, num_timesteps + 1, (batch_routes.shape[0],)).to(device)
        masked_routes = apply_absorbing_mask(batch_routes, t, alphas_cumprod)

        logits = model(masked_routes, t, batch_coords)
        
        loss = criterion(logits.view(-1, 6), batch_routes.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch+1}/{num_epochs} | Average Loss: {total_loss/len(dataloader):.4f}")

    # Save Checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        ckpt_path = f"{save_dir}/kilter_checkpoint_epoch_{epoch+1}.pt"
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': total_loss,
        }
        torch.save(checkpoint, ckpt_path)

